# Advanced Exploratory Data Analysis of NYC Yellow Taxi Trips

## Project scope

This notebook conducts a professional exploratory data analysis of New York City Yellow Taxi trip records. The analysis focuses on demand patterns, economics, data quality, spatial structure, anomalies, and operational segmentation.

The workflow is designed for Google Colab and uses the official monthly Parquet files published by the New York City Taxi & Limousine Commission.

## Analytical objectives

1. Build a reproducible data ingestion and validation layer.
2. Quantify data quality issues before filtering.
3. Analyze temporal demand patterns across days, hours, and weekends.
4. Examine fare, tip, distance, speed, and duration distributions.
5. Enrich trips with taxi zone metadata.
6. Study borough-level and zone-level pickup/drop-off flows.
7. Map spatial demand intensity using official taxi zone geometries.
8. Detect extreme or suspicious trip profiles.
9. Segment pickup zones based on operational behavior.
10. Produce an executive readout grounded in computed metrics.

## Data sources

- NYC TLC Trip Record Data: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- Yellow Taxi data dictionary: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
- Taxi zone lookup table: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
- Taxi zone shapefile: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip

## 1. Environment setup

The notebook uses DuckDB for scalable SQL over Parquet files, pandas for tabular analysis, Plotly for interactive charts, GeoPandas for spatial work, and scikit-learn for anomaly detection and clustering.

In [ ]:
!pip install -q duckdb pyarrow pandas numpy plotly geopandas shapely pyogrio scikit-learn scipy requests tqdm

In [ ]:
from pathlib import Path
import json
import zipfile
import warnings
import requests

import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

px.defaults.template = "plotly_white"

## 2. Configuration

The default configuration analyzes the first quarter of 2024. This is large enough to reveal complex temporal and spatial structure while remaining manageable in a Colab session.

To scale the project up, add more months to `YEAR_MONTHS`. The pipeline is designed to query full Parquet files with DuckDB and sample rows only for high-density visualizations.

In [ ]:
PROJECT_DIR = Path("/content/nyc_yellow_taxi_advanced_eda")
RAW_DIR = PROJECT_DIR / "raw"
GEO_DIR = PROJECT_DIR / "geo"
RAW_DIR.mkdir(parents=True, exist_ok=True)
GEO_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MONTHS = ["2024-01", "2024-02", "2024-03"]

BASE_URL = "https://d37ci6vzurychx.cloudfront.net"
TRIP_URL_TEMPLATE = BASE_URL + "/trip-data/yellow_tripdata_{year_month}.parquet"
ZONE_LOOKUP_URL = BASE_URL + "/misc/taxi_zone_lookup.csv"
ZONE_SHAPEFILE_URL = BASE_URL + "/misc/taxi_zones.zip"

ZONE_LOOKUP_PATH = RAW_DIR / "taxi_zone_lookup.csv"
ZONE_SHAPEFILE_ZIP = GEO_DIR / "taxi_zones.zip"

RANDOM_SEED = 42
SAMPLE_ROWS = 250_000
RUN_GEOSPATIAL_MAPS = True

MIN_DURATION_MIN = 1
MAX_DURATION_MIN = 240
MIN_DISTANCE_MILES = 0.1
MAX_DISTANCE_MILES = 100
MIN_AVG_MPH = 1
MAX_AVG_MPH = 80
MAX_TOTAL_AMOUNT = 500

analysis_start = pd.Period(min(YEAR_MONTHS), freq="M").start_time
analysis_end = (pd.Period(max(YEAR_MONTHS), freq="M") + 1).start_time
ANALYSIS_START_DATE = analysis_start.strftime("%Y-%m-%d")
ANALYSIS_END_DATE = analysis_end.strftime("%Y-%m-%d")

print(f"Analysis window: [{ANALYSIS_START_DATE}, {ANALYSIS_END_DATE})")
print(f"Requested monthly files: {', '.join(YEAR_MONTHS)}")

## 3. Data ingestion

The following cell downloads the monthly Parquet files, taxi zone lookup table, and spatial boundary file. Existing files are reused.

In [ ]:
def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Found existing file: {destination.name}")
        return destination

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(destination, "wb") as file, tqdm(
            total=total, unit="B", unit_scale=True, desc=destination.name
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))
    return destination


trip_files = []
for ym in YEAR_MONTHS:
    path = RAW_DIR / f"yellow_tripdata_{ym}.parquet"
    trip_files.append(download_file(TRIP_URL_TEMPLATE.format(year_month=ym), path))

download_file(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH)
download_file(ZONE_SHAPEFILE_URL, ZONE_SHAPEFILE_ZIP)

print(f"\nDownloaded or found {len(trip_files)} trip files.")

## 4. Analytical layer with DuckDB

DuckDB reads the Parquet files directly and creates:

- a raw view;
- a typed base view;
- a strict clean view;
- an enriched view joined with taxi zone metadata.

The clean layer removes records that are unusable for core operational analysis. The raw layer remains available for profiling and audit checks.

In [ ]:
con = duckdb.connect(database=":memory:")

parquet_list_sql = ", ".join([f"'{path.as_posix()}'" for path in trip_files])

con.execute(f"""
CREATE OR REPLACE VIEW raw_yellow AS
SELECT *
FROM read_parquet([{parquet_list_sql}], union_by_name = true);
""")

zones_df = pd.read_csv(ZONE_LOOKUP_PATH)
zones_df.columns = [col.strip() for col in zones_df.columns]
con.register("zones_df", zones_df)

con.execute("""
CREATE OR REPLACE TABLE zones AS
SELECT
    CAST(LocationID AS INTEGER) AS location_id,
    CAST(Borough AS VARCHAR) AS borough,
    CAST(Zone AS VARCHAR) AS zone,
    CAST(service_zone AS VARCHAR) AS service_zone
FROM zones_df;
""")

con.execute(f"""
CREATE OR REPLACE VIEW taxi_base AS
WITH typed AS (
    SELECT
        TRY_CAST(VendorID AS INTEGER) AS vendor_id,
        CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) AS dropoff_datetime,
        TRY_CAST(passenger_count AS DOUBLE) AS passenger_count,
        TRY_CAST(trip_distance AS DOUBLE) AS trip_distance,
        TRY_CAST(RatecodeID AS INTEGER) AS ratecode_id,
        CAST(store_and_fwd_flag AS VARCHAR) AS store_and_fwd_flag,
        TRY_CAST(PULocationID AS INTEGER) AS pickup_location_id,
        TRY_CAST(DOLocationID AS INTEGER) AS dropoff_location_id,
        TRY_CAST(payment_type AS INTEGER) AS payment_type,
        TRY_CAST(fare_amount AS DOUBLE) AS fare_amount,
        TRY_CAST(extra AS DOUBLE) AS extra,
        TRY_CAST(mta_tax AS DOUBLE) AS mta_tax,
        TRY_CAST(tip_amount AS DOUBLE) AS tip_amount,
        TRY_CAST(tolls_amount AS DOUBLE) AS tolls_amount,
        TRY_CAST(improvement_surcharge AS DOUBLE) AS improvement_surcharge,
        TRY_CAST(total_amount AS DOUBLE) AS total_amount,
        TRY_CAST(congestion_surcharge AS DOUBLE) AS congestion_surcharge,
        TRY_CAST(Airport_fee AS DOUBLE) AS airport_fee
    FROM raw_yellow
),
derived AS (
    SELECT
        *,
        date_diff('second', pickup_datetime, dropoff_datetime) / 60.0 AS duration_min
    FROM typed
)
SELECT
    *,
    trip_distance / NULLIF(duration_min / 60.0, 0) AS avg_mph,
    CASE
        WHEN total_amount > tip_amount AND tip_amount >= 0
        THEN tip_amount / NULLIF(total_amount - tip_amount, 0)
        ELSE NULL
    END AS tip_pct,
    CAST(pickup_datetime AS DATE) AS pickup_date,
    EXTRACT(hour FROM pickup_datetime) AS pickup_hour,
    CAST(strftime(pickup_datetime, '%w') AS INTEGER) AS pickup_dow,
    strftime(pickup_datetime, '%A') AS pickup_day_name,
    CASE WHEN CAST(strftime(pickup_datetime, '%w') AS INTEGER) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,
    CASE
        WHEN EXTRACT(hour FROM pickup_datetime) BETWEEN 7 AND 9
          OR EXTRACT(hour FROM pickup_datetime) BETWEEN 16 AND 19
        THEN 1 ELSE 0
    END AS is_rush_hour
FROM derived;
""")

con.execute(f"""
CREATE OR REPLACE VIEW taxi_clean AS
SELECT *
FROM taxi_base
WHERE pickup_datetime >= TIMESTAMP '{ANALYSIS_START_DATE}'
  AND pickup_datetime < TIMESTAMP '{ANALYSIS_END_DATE}'
  AND dropoff_datetime > pickup_datetime
  AND duration_min BETWEEN {MIN_DURATION_MIN} AND {MAX_DURATION_MIN}
  AND trip_distance BETWEEN {MIN_DISTANCE_MILES} AND {MAX_DISTANCE_MILES}
  AND avg_mph BETWEEN {MIN_AVG_MPH} AND {MAX_AVG_MPH}
  AND fare_amount >= 0
  AND total_amount BETWEEN 0.01 AND {MAX_TOTAL_AMOUNT}
  AND tip_amount >= 0
  AND passenger_count BETWEEN 1 AND 6
  AND pickup_location_id BETWEEN 1 AND 263
  AND dropoff_location_id BETWEEN 1 AND 263;
""")

con.execute("""
CREATE OR REPLACE VIEW taxi_enriched AS
SELECT
    c.*,
    pu.borough AS pickup_borough,
    pu.zone AS pickup_zone,
    pu.service_zone AS pickup_service_zone,
    do.borough AS dropoff_borough,
    do.zone AS dropoff_zone,
    do.service_zone AS dropoff_service_zone,
    CASE
        WHEN pu.zone ILIKE '%Airport%' OR do.zone ILIKE '%Airport%'
        THEN 1 ELSE 0
    END AS touches_airport
FROM taxi_clean c
LEFT JOIN zones pu
    ON c.pickup_location_id = pu.location_id
LEFT JOIN zones do
    ON c.dropoff_location_id = do.location_id;
""")

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

schema = q("DESCRIBE SELECT * FROM raw_yellow")
display(schema)

## 5. Data quality audit

Before interpreting behavior, the notebook quantifies the records filtered out by the clean analytical layer. This step makes assumptions transparent and separates data profiling from business interpretation.

In [ ]:
raw_rows = q("SELECT COUNT(*) AS rows FROM taxi_base").loc[0, "rows"]
clean_rows = q("SELECT COUNT(*) AS rows FROM taxi_clean").loc[0, "rows"]

qa = q(f"""
SELECT
    COUNT(*) AS raw_rows,
    SUM(CASE WHEN pickup_datetime < TIMESTAMP '{ANALYSIS_START_DATE}' OR pickup_datetime >= TIMESTAMP '{ANALYSIS_END_DATE}' THEN 1 ELSE 0 END) AS outside_requested_window,
    SUM(CASE WHEN dropoff_datetime <= pickup_datetime THEN 1 ELSE 0 END) AS non_positive_duration,
    SUM(CASE WHEN duration_min NOT BETWEEN {MIN_DURATION_MIN} AND {MAX_DURATION_MIN} THEN 1 ELSE 0 END) AS implausible_duration,
    SUM(CASE WHEN trip_distance NOT BETWEEN {MIN_DISTANCE_MILES} AND {MAX_DISTANCE_MILES} THEN 1 ELSE 0 END) AS implausible_distance,
    SUM(CASE WHEN avg_mph NOT BETWEEN {MIN_AVG_MPH} AND {MAX_AVG_MPH} THEN 1 ELSE 0 END) AS implausible_speed,
    SUM(CASE WHEN fare_amount < 0 OR total_amount NOT BETWEEN 0.01 AND {MAX_TOTAL_AMOUNT} OR tip_amount < 0 THEN 1 ELSE 0 END) AS implausible_amount,
    SUM(CASE WHEN passenger_count NOT BETWEEN 1 AND 6 OR passenger_count IS NULL THEN 1 ELSE 0 END) AS implausible_passenger_count,
    SUM(CASE WHEN pickup_location_id NOT BETWEEN 1 AND 263 OR dropoff_location_id NOT BETWEEN 1 AND 263 THEN 1 ELSE 0 END) AS invalid_location_id
FROM taxi_base;
""")

retention = clean_rows / raw_rows
display(qa.T.rename(columns={0: "rows"}))

display(Markdown(f"""
### Data quality readout

- Raw records loaded: **{raw_rows:,.0f}**
- Clean analytical records retained: **{clean_rows:,.0f}**
- Retention rate: **{retention:.2%}**

Records are filtered for temporal consistency, valid trip duration, meaningful distance, plausible speed, non-negative financial values, usable passenger counts, and valid TLC location identifiers.
"""))

In [ ]:
critical_columns = [
    "passenger_count", "trip_distance", "ratecode_id", "pickup_location_id",
    "dropoff_location_id", "payment_type", "fare_amount", "tip_amount",
    "total_amount", "congestion_surcharge", "airport_fee"
]

missing_frames = []
for col in critical_columns:
    missing_frames.append(q(f"""
    SELECT
        '{col}' AS column_name,
        SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS missing_rows,
        COUNT(*) AS total_rows
    FROM taxi_base
    """))

missing_df = pd.concat(missing_frames, ignore_index=True)
missing_df["missing_pct"] = missing_df["missing_rows"] / missing_df["total_rows"]

display(missing_df.sort_values("missing_pct", ascending=False))

plot_missing = missing_df.sort_values("missing_pct", ascending=True).copy()
plot_missing["label"] = plot_missing["missing_pct"].map(lambda x: f"{x:.2%}")

fig = px.bar(
    plot_missing,
    x="missing_pct",
    y="column_name",
    orientation="h",
    title="Missingness across critical fields",
    labels={"missing_pct": "Missing share", "column_name": "Field"},
    text="label"
)
fig.update_layout(height=500)
fig.update_xaxes(tickformat=".1%")
fig.show()

## 6. Core descriptive overview

This section builds a high-level operational picture of the cleaned dataset: volume, revenue, distance, duration, speed, fares, and tips.

In [ ]:
overview = q("""
SELECT
    MIN(pickup_datetime) AS first_pickup,
    MAX(pickup_datetime) AS last_pickup,
    COUNT(*) AS trips,
    COUNT(DISTINCT pickup_date) AS active_days,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    MEDIAN(total_amount) AS median_total_amount,
    AVG(trip_distance) AS avg_distance_miles,
    MEDIAN(trip_distance) AS median_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    MEDIAN(duration_min) AS median_duration_min,
    AVG(avg_mph) AS avg_mph,
    MEDIAN(avg_mph) AS median_mph,
    AVG(tip_pct) AS avg_tip_pct,
    MEDIAN(tip_pct) AS median_tip_pct
FROM taxi_enriched;
""")

display(overview.T.rename(columns={0: "value"}))

In [ ]:
sample_df = q(f"""
SELECT
    pickup_datetime,
    pickup_date,
    pickup_hour,
    pickup_day_name,
    is_weekend,
    is_rush_hour,
    passenger_count,
    trip_distance,
    duration_min,
    avg_mph,
    fare_amount,
    tip_amount,
    total_amount,
    tip_pct,
    payment_type,
    pickup_borough,
    pickup_zone,
    dropoff_borough,
    dropoff_zone,
    touches_airport
FROM taxi_enriched
USING SAMPLE reservoir({SAMPLE_ROWS} ROWS) REPEATABLE ({RANDOM_SEED});
""")

print(f"Sample rows for dense visualizations: {len(sample_df):,}")
display(sample_df.head())

## 7. Distributional analysis

Taxi trip variables are strongly skewed. The plots below use clipped x-axis ranges based on empirical quantiles so that central structure remains readable while tail behavior is still measured numerically.

In [ ]:
def quantile_range(frame: pd.DataFrame, column: str, lower: float = 0.001, upper: float = 0.995):
    values = frame[column].replace([np.inf, -np.inf], np.nan).dropna()
    return values.quantile(lower), values.quantile(upper)

dist_summary = sample_df[["trip_distance", "duration_min", "avg_mph", "total_amount", "tip_pct"]].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).T

display(dist_summary)

In [ ]:
for column, label in [
    ("trip_distance", "Trip distance, miles"),
    ("duration_min", "Trip duration, minutes"),
    ("avg_mph", "Average speed, mph"),
    ("total_amount", "Total amount, USD"),
    ("tip_pct", "Tip percentage")
]:
    low, high = quantile_range(sample_df, column)
    plot_df = sample_df[(sample_df[column] >= low) & (sample_df[column] <= high)].copy()

    fig = px.histogram(
        plot_df,
        x=column,
        nbins=80,
        marginal="box",
        title=f"Distribution of {label}",
        labels={column: label}
    )
    fig.update_layout(height=520, bargap=0.02)
    fig.show()

### Distributional considerations

- Distance, duration, fare, and tip variables usually show heavy right tails.
- The median is often more representative than the mean for trip-level behavior.
- Cash tips are generally not fully observed in the same way as card tips, so tip analysis should be interpreted as recorded tip behavior rather than complete passenger tipping behavior.
- Speed is a derived metric and is especially sensitive to timestamp, distance, and duration errors.

## 8. Temporal demand structure

This section studies daily demand, hourly rhythms, weekend effects, and rush-hour patterns.

In [ ]:
daily = q("""
SELECT
    pickup_date,
    COUNT(*) AS trips,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    AVG(avg_mph) AS avg_mph,
    AVG(duration_min) AS avg_duration_min,
    AVG(tip_pct) AS avg_tip_pct
FROM taxi_enriched
GROUP BY pickup_date
ORDER BY pickup_date;
""")

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily["pickup_date"], y=daily["trips"], mode="lines", name="Trips"))
fig.add_trace(go.Scatter(
    x=daily["pickup_date"],
    y=daily["gross_booking_value"],
    mode="lines",
    name="Gross booking value",
    yaxis="y2"
))
fig.update_layout(
    title="Daily trips and gross booking value",
    xaxis_title="Pickup date",
    yaxis=dict(title="Trips"),
    yaxis2=dict(title="Gross booking value, USD", overlaying="y", side="right"),
    height=520
)
fig.show()

display(daily.describe().T)

In [ ]:
hour_day = q("""
SELECT
    pickup_day_name,
    pickup_dow,
    pickup_hour,
    COUNT(*) AS trips,
    AVG(avg_mph) AS avg_mph,
    AVG(total_amount) AS avg_total_amount,
    AVG(tip_pct) AS avg_tip_pct
FROM taxi_enriched
GROUP BY pickup_day_name, pickup_dow, pickup_hour
ORDER BY pickup_dow, pickup_hour;
""")

day_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

heatmap_trips = (
    hour_day.pivot(index="pickup_day_name", columns="pickup_hour", values="trips")
    .reindex(day_order)
    .fillna(0)
)

fig = px.imshow(
    heatmap_trips,
    aspect="auto",
    title="Trip volume by day of week and pickup hour",
    labels=dict(x="Pickup hour", y="Day of week", color="Trips")
)
fig.update_layout(height=520)
fig.show()

heatmap_speed = (
    hour_day.pivot(index="pickup_day_name", columns="pickup_hour", values="avg_mph")
    .reindex(day_order)
)

fig = px.imshow(
    heatmap_speed,
    aspect="auto",
    title="Average speed by day of week and pickup hour",
    labels=dict(x="Pickup hour", y="Day of week", color="Average mph")
)
fig.update_layout(height=520)
fig.show()

In [ ]:
rush = q("""
SELECT
    CASE WHEN is_rush_hour = 1 THEN 'Rush hour' ELSE 'Non-rush hour' END AS period_type,
    COUNT(*) AS trips,
    AVG(duration_min) AS avg_duration_min,
    MEDIAN(duration_min) AS median_duration_min,
    AVG(avg_mph) AS avg_mph,
    MEDIAN(avg_mph) AS median_mph,
    AVG(total_amount) AS avg_total_amount
FROM taxi_enriched
GROUP BY period_type
ORDER BY period_type;
""")

display(rush)

rush_plot = sample_df.copy()
rush_plot["period_type"] = rush_plot["is_rush_hour"].map({1: "Rush hour", 0: "Non-rush hour"})

fig = px.box(
    rush_plot,
    x="period_type",
    y="avg_mph",
    points=False,
    title="Speed distribution during rush and non-rush periods",
    labels={"period_type": "Period type", "avg_mph": "Average mph"}
)
fig.update_layout(height=520)
fig.show()

### Temporal considerations

- Demand should be evaluated jointly with average revenue and average speed. A busy hour is not necessarily the most efficient hour.
- Rush-hour periods often show a trade-off between high demand and slower realized speed.
- Weekend patterns can differ sharply from weekday commuter patterns, especially late evening and airport-connected trips.

## 9. Fare, tip, and payment behavior

This section separates trip economics from trip volume. It evaluates total amount, recorded tips, payment method, and airport-related trips.

In [ ]:
payment_map = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}
sample_df["payment_label"] = sample_df["payment_type"].map(payment_map).fillna("Other")

payment_summary = q("""
SELECT
    payment_type,
    COUNT(*) AS trips,
    AVG(total_amount) AS avg_total_amount,
    MEDIAN(total_amount) AS median_total_amount,
    AVG(tip_amount) AS avg_tip_amount,
    MEDIAN(tip_amount) AS median_tip_amount,
    AVG(tip_pct) AS avg_tip_pct,
    MEDIAN(tip_pct) AS median_tip_pct
FROM taxi_enriched
GROUP BY payment_type
ORDER BY trips DESC;
""")

payment_summary["payment_label"] = payment_summary["payment_type"].map(payment_map).fillna("Other")
display(payment_summary)

fig = px.bar(
    payment_summary.sort_values("trips", ascending=True),
    x="trips",
    y="payment_label",
    orientation="h",
    title="Trip volume by payment type",
    labels={"trips": "Trips", "payment_label": "Payment type"}
)
fig.update_layout(height=420)
fig.show()

tip_plot = sample_df[
    (sample_df["payment_label"].isin(["Credit card", "Cash"]))
    & (sample_df["tip_pct"].between(0, sample_df["tip_pct"].quantile(0.995), inclusive="both"))
].copy()

fig = px.violin(
    tip_plot,
    x="payment_label",
    y="tip_pct",
    box=True,
    points=False,
    title="Recorded tip percentage by payment type",
    labels={"payment_label": "Payment type", "tip_pct": "Tip percentage"}
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=520)
fig.show()

In [ ]:
airport_summary = q("""
SELECT
    CASE WHEN touches_airport = 1 THEN 'Touches airport zone' ELSE 'Non-airport trip' END AS airport_flag,
    COUNT(*) AS trips,
    AVG(trip_distance) AS avg_distance_miles,
    MEDIAN(trip_distance) AS median_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    MEDIAN(duration_min) AS median_duration_min,
    AVG(total_amount) AS avg_total_amount,
    MEDIAN(total_amount) AS median_total_amount,
    AVG(tip_pct) AS avg_tip_pct,
    MEDIAN(tip_pct) AS median_tip_pct
FROM taxi_enriched
GROUP BY airport_flag;
""")

display(airport_summary)

airport_plot = sample_df.copy()
airport_plot["airport_flag"] = airport_plot["touches_airport"].map({1: "Touches airport zone", 0: "Non-airport trip"})

fig = px.box(
    airport_plot,
    x="airport_flag",
    y="total_amount",
    points=False,
    title="Total amount distribution for airport and non-airport trips",
    labels={"airport_flag": "", "total_amount": "Total amount, USD"}
)
fig.update_yaxes(range=[0, sample_df["total_amount"].quantile(0.995)])
fig.update_layout(height=520)
fig.show()

### Economic considerations

- Airport-connected trips tend to have different distance, duration, fare, and tip profiles from dense urban trips.
- Payment type affects observability. Recorded card tips should not be treated as a complete estimate of all tipping behavior.
- Average gross booking value can be pulled upward by long trips and airport trips, so median and quantile analysis are essential.

## 10. Spatial enrichment and flow analysis

Trips are joined to TLC taxi zones so that pickup and drop-off locations can be analyzed at borough and zone level.

In [ ]:
borough_summary = q("""
SELECT
    pickup_borough,
    COUNT(*) AS pickups,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    AVG(trip_distance) AS avg_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    AVG(avg_mph) AS avg_mph,
    AVG(tip_pct) AS avg_tip_pct
FROM taxi_enriched
GROUP BY pickup_borough
ORDER BY pickups DESC;
""")

display(borough_summary)

fig = px.bar(
    borough_summary.sort_values("pickups", ascending=True),
    x="pickups",
    y="pickup_borough",
    orientation="h",
    title="Pickups by borough",
    labels={"pickups": "Pickups", "pickup_borough": "Pickup borough"}
)
fig.update_layout(height=420)
fig.show()

In [ ]:
od_borough = q("""
SELECT
    pickup_borough,
    dropoff_borough,
    COUNT(*) AS trips,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount
FROM taxi_enriched
WHERE pickup_borough IS NOT NULL
  AND dropoff_borough IS NOT NULL
GROUP BY pickup_borough, dropoff_borough
ORDER BY trips DESC;
""")

od_matrix = (
    od_borough.pivot(index="pickup_borough", columns="dropoff_borough", values="trips")
    .fillna(0)
)

fig = px.imshow(
    od_matrix,
    aspect="auto",
    text_auto=".2s",
    title="Borough-to-borough trip matrix",
    labels=dict(x="Drop-off borough", y="Pickup borough", color="Trips")
)
fig.update_layout(height=560)
fig.show()

display(od_borough.head(20))

In [ ]:
top_zones = q("""
SELECT
    pickup_borough,
    pickup_zone,
    COUNT(*) AS pickups,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    AVG(avg_mph) AS avg_mph,
    AVG(tip_pct) AS avg_tip_pct
FROM taxi_enriched
WHERE pickup_zone IS NOT NULL
GROUP BY pickup_borough, pickup_zone
ORDER BY pickups DESC
LIMIT 25;
""")

display(top_zones)

fig = px.bar(
    top_zones.sort_values("pickups", ascending=True),
    x="pickups",
    y="pickup_zone",
    color="pickup_borough",
    orientation="h",
    title="Top pickup zones",
    labels={"pickups": "Pickups", "pickup_zone": "Pickup zone", "pickup_borough": "Borough"}
)
fig.update_layout(height=720)
fig.show()

In [ ]:
zone_balance = q("""
WITH pickups AS (
    SELECT pickup_location_id AS location_id, COUNT(*) AS pickups
    FROM taxi_enriched
    GROUP BY pickup_location_id
),
dropoffs AS (
    SELECT dropoff_location_id AS location_id, COUNT(*) AS dropoffs
    FROM taxi_enriched
    GROUP BY dropoff_location_id
)
SELECT
    z.borough,
    z.zone,
    COALESCE(p.pickups, 0) AS pickups,
    COALESCE(d.dropoffs, 0) AS dropoffs,
    COALESCE(p.pickups, 0) - COALESCE(d.dropoffs, 0) AS net_pickup_pressure
FROM zones z
LEFT JOIN pickups p ON z.location_id = p.location_id
LEFT JOIN dropoffs d ON z.location_id = d.location_id
ORDER BY ABS(net_pickup_pressure) DESC;
""")

balance_plot = pd.concat([
    zone_balance.nlargest(15, "net_pickup_pressure"),
    zone_balance.nsmallest(15, "net_pickup_pressure")
]).sort_values("net_pickup_pressure")

fig = px.bar(
    balance_plot,
    x="net_pickup_pressure",
    y="zone",
    color="borough",
    orientation="h",
    title="Zones with strongest pickup/drop-off imbalance",
    labels={"net_pickup_pressure": "Pickups minus drop-offs", "zone": "Taxi zone"}
)
fig.update_layout(height=760)
fig.show()

display(zone_balance.head(30))

### Spatial considerations

- Pickup volume, revenue, and net pickup pressure measure different aspects of a zone.
- Net pickup pressure can suggest directional flow imbalance, but it is not by itself a supply shortage estimate.
- Borough matrices are useful for macro patterns, while zone-level analysis reveals operational microstructure.

## 11. Geospatial choropleth maps

The following section reads the official TLC taxi zone boundary file and maps zone-level metrics. It can be disabled by setting `RUN_GEOSPATIAL_MAPS = False`.

In [ ]:
if RUN_GEOSPATIAL_MAPS:
    import geopandas as gpd

    extract_dir = GEO_DIR / "taxi_zones"
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not any(extract_dir.rglob("*.shp")):
        with zipfile.ZipFile(ZONE_SHAPEFILE_ZIP, "r") as archive:
            archive.extractall(extract_dir)

    shp_path = next(extract_dir.rglob("*.shp"))
    zones_gdf = gpd.read_file(shp_path).to_crs(4326)
    zones_gdf["LocationID_str"] = zones_gdf["LocationID"].astype(int).astype(str)

    zone_metrics = q("""
    SELECT
        pickup_location_id AS location_id,
        MAX(pickup_borough) AS borough,
        MAX(pickup_zone) AS zone,
        COUNT(*) AS pickups,
        SUM(total_amount) AS gross_booking_value,
        AVG(total_amount) AS avg_total_amount,
        AVG(avg_mph) AS avg_mph,
        AVG(tip_pct) AS avg_tip_pct
    FROM taxi_enriched
    GROUP BY pickup_location_id;
    """)

    zone_metrics["LocationID_str"] = zone_metrics["location_id"].astype(int).astype(str)

    map_gdf = zones_gdf.merge(
        zone_metrics,
        on="LocationID_str",
        how="left"
    )

    map_gdf["pickups"] = map_gdf["pickups"].fillna(0)
    map_gdf["log_pickups"] = np.log1p(map_gdf["pickups"])
    map_gdf["gross_booking_value"] = map_gdf["gross_booking_value"].fillna(0)

    geojson = json.loads(map_gdf.to_json())
    for feature in geojson["features"]:
        feature["id"] = feature["properties"]["LocationID_str"]

    fig = px.choropleth_mapbox(
        map_gdf,
        geojson=geojson,
        locations="LocationID_str",
        color="log_pickups",
        hover_name="zone",
        hover_data={
            "borough": True,
            "pickups": ":,.0f",
            "gross_booking_value": ":,.0f",
            "avg_total_amount": ":.2f",
            "avg_mph": ":.1f",
            "LocationID_str": False,
            "log_pickups": False
        },
        mapbox_style="carto-positron",
        center={"lat": 40.7128, "lon": -74.0060},
        zoom=9.4,
        opacity=0.68,
        title="Pickup intensity by TLC taxi zone"
    )
    fig.update_layout(height=720, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    fig.show()

    fig = px.choropleth_mapbox(
        map_gdf,
        geojson=geojson,
        locations="LocationID_str",
        color="avg_total_amount",
        hover_name="zone",
        hover_data={
            "borough": True,
            "pickups": ":,.0f",
            "gross_booking_value": ":,.0f",
            "avg_total_amount": ":.2f",
            "avg_mph": ":.1f",
            "LocationID_str": False
        },
        mapbox_style="carto-positron",
        center={"lat": 40.7128, "lon": -74.0060},
        zoom=9.4,
        opacity=0.68,
        title="Average total amount by pickup zone"
    )
    fig.update_layout(height=720, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    fig.show()
else:
    print("Geospatial maps are disabled.")

## 12. Origin-destination corridor analysis

This section identifies the highest-volume zone-to-zone corridors and evaluates whether the market is dominated by a small number of routes.

In [ ]:
corridors = q("""
SELECT
    pickup_borough,
    pickup_zone,
    dropoff_borough,
    dropoff_zone,
    COUNT(*) AS trips,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    AVG(trip_distance) AS avg_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    AVG(avg_mph) AS avg_mph
FROM taxi_enriched
WHERE pickup_zone IS NOT NULL
  AND dropoff_zone IS NOT NULL
GROUP BY pickup_borough, pickup_zone, dropoff_borough, dropoff_zone
ORDER BY trips DESC
LIMIT 30;
""")

corridors["route"] = corridors["pickup_zone"] + " → " + corridors["dropoff_zone"]
display(corridors)

fig = px.bar(
    corridors.sort_values("trips", ascending=True),
    x="trips",
    y="route",
    color="pickup_borough",
    orientation="h",
    title="Top origin-destination taxi corridors",
    labels={"trips": "Trips", "route": "Route", "pickup_borough": "Pickup borough"}
)
fig.update_layout(height=860)
fig.show()

In [ ]:
route_concentration = q("""
WITH routes AS (
    SELECT
        pickup_location_id,
        dropoff_location_id,
        COUNT(*) AS trips
    FROM taxi_enriched
    GROUP BY pickup_location_id, dropoff_location_id
),
ranked AS (
    SELECT
        *,
        SUM(trips) OVER () AS total_trips,
        SUM(trips) OVER (ORDER BY trips DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_trips,
        ROW_NUMBER() OVER (ORDER BY trips DESC) AS route_rank
    FROM routes
)
SELECT
    route_rank,
    trips,
    cumulative_trips / total_trips AS cumulative_trip_share
FROM ranked
ORDER BY route_rank
LIMIT 500;
""")

fig = px.line(
    route_concentration,
    x="route_rank",
    y="cumulative_trip_share",
    title="Cumulative trip concentration across top 500 origin-destination routes",
    labels={"route_rank": "Route rank", "cumulative_trip_share": "Cumulative share of all trips"}
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=500)
fig.show()

display(route_concentration.head(20))

### Corridor considerations

- Concentration analysis helps determine whether demand is broad-based or concentrated in a limited set of corridors.
- Route ranking is sensitive to zone granularity. TLC zones are not equal in population, land area, or commercial activity.
- High-volume routes may be operationally important even if their average fare is not the highest.

## 13. Anomaly detection

This section combines rule-based filtering with unsupervised anomaly detection. The goal is not to label fraud, but to surface trip profiles that deserve inspection.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

anomaly_features = ["duration_min", "trip_distance", "avg_mph", "total_amount", "tip_pct"]
model_df = sample_df[anomaly_features].replace([np.inf, -np.inf], np.nan).dropna().copy()

for col in anomaly_features:
    lower = model_df[col].quantile(0.001)
    upper = model_df[col].quantile(0.999)
    model_df[col] = model_df[col].clip(lower, upper)

scaler = RobustScaler()
X = scaler.fit_transform(model_df)

model = IsolationForest(
    n_estimators=300,
    contamination=0.015,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
model_df["anomaly_score"] = -model.fit(X).score_samples(X)
threshold = model_df["anomaly_score"].quantile(0.985)
model_df["is_anomaly"] = model_df["anomaly_score"] >= threshold

sample_anomaly = sample_df.loc[model_df.index].copy()
sample_anomaly["anomaly_score"] = model_df["anomaly_score"]
sample_anomaly["is_anomaly"] = model_df["is_anomaly"]

print(f"Anomaly threshold: {threshold:.4f}")
print(f"Flagged anomalies in visualization sample: {sample_anomaly['is_anomaly'].sum():,}")

plot_anomaly = sample_anomaly.sample(min(60_000, len(sample_anomaly)), random_state=RANDOM_SEED)

fig = px.scatter(
    plot_anomaly,
    x="duration_min",
    y="trip_distance",
    color="is_anomaly",
    hover_data=["avg_mph", "total_amount", "tip_pct", "pickup_borough", "pickup_zone", "dropoff_zone"],
    title="Distance-duration profile with unsupervised anomaly flags",
    labels={"duration_min": "Duration, minutes", "trip_distance": "Distance, miles", "is_anomaly": "Flagged anomaly"}
)
fig.update_layout(height=620)
fig.show()

top_anomalies = sample_anomaly.sort_values("anomaly_score", ascending=False).head(25)
display(top_anomalies[
    ["pickup_datetime", "pickup_borough", "pickup_zone", "dropoff_borough", "dropoff_zone",
     "duration_min", "trip_distance", "avg_mph", "total_amount", "tip_pct", "anomaly_score"]
])

### Anomaly considerations

- Isolation Forest highlights unusual combinations of features, not necessarily invalid records.
- A trip can be statistically unusual because it is long, short, expensive, slow, fast, or economically atypical.
- Operational review should combine anomaly scores with domain rules and map context.

## 14. Zone segmentation

Pickup zones are grouped by operational behavior. The segmentation uses trip count, average fare, speed, distance, duration, tip behavior, and airport exposure.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

zone_features = q("""
SELECT
    pickup_location_id AS location_id,
    MAX(pickup_borough) AS borough,
    MAX(pickup_zone) AS zone,
    COUNT(*) AS trips,
    AVG(total_amount) AS avg_total_amount,
    AVG(trip_distance) AS avg_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    AVG(avg_mph) AS avg_mph,
    AVG(tip_pct) AS avg_tip_pct,
    AVG(touches_airport) AS airport_touch_share
FROM taxi_enriched
GROUP BY pickup_location_id
HAVING COUNT(*) >= 500
ORDER BY trips DESC;
""")

feature_cols = [
    "trips",
    "avg_total_amount",
    "avg_distance_miles",
    "avg_duration_min",
    "avg_mph",
    "avg_tip_pct",
    "airport_touch_share"
]

seg = zone_features.dropna(subset=feature_cols).copy()
seg_model = seg[feature_cols].copy()
seg_model["trips"] = np.log1p(seg_model["trips"])

scaler = StandardScaler()
X_seg = scaler.fit_transform(seg_model)

kmeans = KMeans(n_clusters=5, random_state=RANDOM_SEED, n_init=20)
seg["cluster"] = kmeans.fit_predict(X_seg).astype(str)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords = pca.fit_transform(X_seg)
seg["pc1"] = coords[:, 0]
seg["pc2"] = coords[:, 1]

cluster_profile = (
    seg.groupby("cluster")[feature_cols]
    .median()
    .sort_values("trips", ascending=False)
)

display(cluster_profile)
display(seg.sort_values("trips", ascending=False).head(25))

fig = px.scatter(
    seg,
    x="pc1",
    y="pc2",
    size="trips",
    color="cluster",
    hover_name="zone",
    hover_data=["borough", "avg_total_amount", "avg_distance_miles", "avg_mph", "airport_touch_share"],
    title="Pickup zone segmentation using PCA projection",
    labels={"pc1": "Principal component 1", "pc2": "Principal component 2"}
)
fig.update_layout(height=620)
fig.show()

In [ ]:
if RUN_GEOSPATIAL_MAPS:
    seg_map = map_gdf.merge(
        seg[["location_id", "cluster"]],
        left_on="LocationID",
        right_on="location_id",
        how="left"
    )
    seg_map["cluster"] = seg_map["cluster"].fillna("Insufficient volume")
    seg_map["LocationID_str"] = seg_map["LocationID"].astype(int).astype(str)

    seg_geojson = json.loads(seg_map.to_json())
    for feature in seg_geojson["features"]:
        feature["id"] = feature["properties"]["LocationID_str"]

    fig = px.choropleth_mapbox(
        seg_map,
        geojson=seg_geojson,
        locations="LocationID_str",
        color="cluster",
        hover_name="zone",
        hover_data={"borough": True, "LocationID_str": False},
        mapbox_style="carto-positron",
        center={"lat": 40.7128, "lon": -74.0060},
        zoom=9.4,
        opacity=0.68,
        title="Pickup zone behavioral clusters"
    )
    fig.update_layout(height=720, margin={"r": 0, "t": 50, "l": 0, "b": 0})
    fig.show()

### Segmentation considerations

- Clusters are exploratory and depend on the selected period, volume threshold, and feature set.
- Airport-connected zones can dominate distance and fare signals.
- A production segmentation would require stability checks across months and sensitivity analysis over the number of clusters.

## 15. Exploratory inference: rush-hour speed effect

This section estimates the difference in median speed between rush-hour and non-rush-hour trips using bootstrap resampling on the visualization sample.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

rush_speed = sample_df.loc[sample_df["is_rush_hour"] == 1, "avg_mph"].dropna().to_numpy()
non_rush_speed = sample_df.loc[sample_df["is_rush_hour"] == 0, "avg_mph"].dropna().to_numpy()

n_boot = 2_000
boot_diffs = np.empty(n_boot)

for i in range(n_boot):
    rush_sample = rng.choice(rush_speed, size=min(20_000, len(rush_speed)), replace=True)
    non_rush_sample = rng.choice(non_rush_speed, size=min(20_000, len(non_rush_speed)), replace=True)
    boot_diffs[i] = np.median(rush_sample) - np.median(non_rush_sample)

ci_low, ci_high = np.quantile(boot_diffs, [0.025, 0.975])
point_estimate = np.median(rush_speed) - np.median(non_rush_speed)

display(Markdown(f"""
### Rush-hour median speed difference

Estimated difference in median speed:

**rush hour minus non-rush hour = {point_estimate:.2f} mph**

Bootstrap 95% interval:

**[{ci_low:.2f}, {ci_high:.2f}] mph**

This is an exploratory estimate, not a causal claim.
"""))

fig = px.histogram(
    pd.DataFrame({"bootstrap_difference_mph": boot_diffs}),
    x="bootstrap_difference_mph",
    nbins=80,
    title="Bootstrap distribution of rush-hour median speed difference",
    labels={"bootstrap_difference_mph": "Median speed difference, mph"}
)
fig.add_vline(x=point_estimate, line_dash="dash")
fig.update_layout(height=500)
fig.show()

## 16. Executive readout

The final cell computes a concise, data-driven readout from the completed analytical layer.

In [ ]:
overview_row = q("""
SELECT
    COUNT(*) AS trips,
    MIN(pickup_date) AS first_date,
    MAX(pickup_date) AS last_date,
    SUM(total_amount) AS gross_booking_value,
    AVG(total_amount) AS avg_total_amount,
    MEDIAN(total_amount) AS median_total_amount,
    AVG(trip_distance) AS avg_distance_miles,
    MEDIAN(trip_distance) AS median_distance_miles,
    AVG(duration_min) AS avg_duration_min,
    MEDIAN(duration_min) AS median_duration_min,
    AVG(avg_mph) AS avg_mph,
    MEDIAN(avg_mph) AS median_mph
FROM taxi_enriched;
""").iloc[0]

busiest_borough = q("""
SELECT pickup_borough, COUNT(*) AS trips
FROM taxi_enriched
GROUP BY pickup_borough
ORDER BY trips DESC
LIMIT 1;
""").iloc[0]

top_route = q("""
SELECT
    pickup_zone,
    dropoff_zone,
    COUNT(*) AS trips
FROM taxi_enriched
GROUP BY pickup_zone, dropoff_zone
ORDER BY trips DESC
LIMIT 1;
""").iloc[0]

busiest_hour = q("""
SELECT pickup_hour, COUNT(*) AS trips
FROM taxi_enriched
GROUP BY pickup_hour
ORDER BY trips DESC
LIMIT 1;
""").iloc[0]

airport_share = q("""
SELECT AVG(touches_airport) AS airport_touch_share
FROM taxi_enriched;
""").iloc[0, 0]

speed_rush = q("""
SELECT
    CASE WHEN is_rush_hour = 1 THEN 'rush' ELSE 'non_rush' END AS period,
    MEDIAN(avg_mph) AS median_mph
FROM taxi_enriched
GROUP BY period;
""")

rush_median = float(speed_rush.loc[speed_rush["period"] == "rush", "median_mph"].iloc[0])
non_rush_median = float(speed_rush.loc[speed_rush["period"] == "non_rush", "median_mph"].iloc[0])

display(Markdown(f"""
### Executive summary

The analytical layer covers **{overview_row['trips']:,.0f} cleaned Yellow Taxi trips** from **{overview_row['first_date']}** to **{overview_row['last_date']}**, with an observed gross booking value of approximately **${overview_row['gross_booking_value']:,.0f}**.

Key findings:

- The busiest pickup borough is **{busiest_borough['pickup_borough']}**, representing **{busiest_borough['trips']:,.0f}** trips in the cleaned layer.
- The busiest pickup hour is **{int(busiest_hour['pickup_hour']):02d}:00**, with **{busiest_hour['trips']:,.0f}** trips.
- The most frequent observed zone corridor is **{top_route['pickup_zone']} → {top_route['dropoff_zone']}**, with **{top_route['trips']:,.0f}** trips.
- The median trip is **{overview_row['median_distance_miles']:.2f} miles**, lasts **{overview_row['median_duration_min']:.1f} minutes**, and has a median total amount of **${overview_row['median_total_amount']:.2f}**.
- Trips touching an airport zone represent **{airport_share:.2%}** of cleaned trips.
- Median speed is **{rush_median:.1f} mph** during rush hours and **{non_rush_median:.1f} mph** outside rush hours.

Recommended next steps:

1. Extend the analysis to a full year and compare monthly seasonality.
2. Add weather, public holidays, major events, and subway disruption data.
3. Build a predictive demand model by zone and hour.
4. Validate anomaly rules with domain-specific trip policies.
5. Re-run segmentation over multiple periods to check cluster stability.
"""))

## 17. Reproducibility notes

- The notebook uses the official TLC public files and can be rerun by changing `YEAR_MONTHS`.
- Dense plots use a reproducible reservoir sample to keep visualization responsive.
- Aggregations, maps, and executive metrics are computed from the cleaned analytical layer.
- The clean layer is intentionally strict. Sensitivity analysis should be performed before using results for operational decisions.